# Lab 5, Day 2 — Pipeline, Features, and Model Selection

Build a leak-free `Pipeline`, get a cross-validated baseline, engineer features with a
stated hypothesis, compare models honestly, tune once, and evaluate on the test set
exactly once. See `Lab5_Day2_Instructions.md` for the full walkthrough.

This continues directly from Day 1's folder and split - not a restart.

## Before you start: watch leakage happen

Fit a `StandardScaler` on your *full* dataset (`X`, before any split) and print
`scaler.mean_`. Then fit a fresh one on `X_train` alone and print its `.mean_`. The
numbers differ. Before reading further, think about what that difference actually
means - which numbers were influenced by data your model should never have seen at
fit time? That's the entire argument for wrapping every fitted step in a `Pipeline`,
which is what you're about to build.

In [1]:
# TODO: the leakage demo described above (optional to keep in your final notebook,
# but do it before writing any pipeline code)
import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import (StratifiedKFold,cross_val_score,GridSearchCV)
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

from sklearn.metrics import (f1_score, classification_report)



In [2]:

# TODO: reload yesterday's split with joblib.load("split.joblib"), or re-run Day 1's
# Step 5-6 if you didn't save it

split = joblib.load("split.joblib")

X_train = split["X_train"]
X_test = split["X_test"]
y_train = split["y_train"]
y_test = split["y_test"]

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

X_train: (1047, 8)
X_test : (262, 8)
y_train: (1047,)
y_test : (262,)


In [3]:
# Combine train and test ONLY for demonstrating leakage
X_full = pd.concat([X_train, X_test])
numeric_cols_demo = [
    c for c in X_train.select_dtypes(include=np.number).columns
    if c != "Pclass"
]

scaler_full = StandardScaler()
scaler_full.fit(X_full[numeric_cols_demo])

scaler_train = StandardScaler()
scaler_train.fit(X_train[numeric_cols_demo])

print("Means learned from FULL dataset:")
print(pd.Series(scaler_full.mean_,index=numeric_cols_demo))

print("\nMeans learned from TRAINING dataset:")
print(pd.Series(scaler_train.mean_,index=numeric_cols_demo))

print("\nAre the means identical?\n",np.allclose(scaler_full.mean_,scaler_train.mean_))

Means learned from FULL dataset:
Age            29.881135
SibSp           0.498854
Parch           0.385027
Fare           33.295479
Cabin_known     0.225363
dtype: float64

Means learned from TRAINING dataset:
Age            29.604316
SibSp           0.484241
Parch           0.385864
Fare           32.166838
Cabin_known     0.214900
dtype: float64

Are the means identical?
 False


### Leakage Demonstration

The means learned by the scaler fitted on the full dataset are different from those learned using only the training data. For example, the mean of `Age` and `Fare`,etc. changes when the test set is included.

This demonstrates that fitting preprocessing on the full dataset would allow information from the test set to influence the preprocessing step. To avoid this leakage, preprocessing should be placed inside a `Pipeline`, so that it is fitted only on the training portion of each cross-validation fold.


## Step 1: The ColumnTransformer (`pipeline.py`)

In [4]:
# TODO: build_preprocessor(num_cols, cat_cols) - a numeric sub-pipeline (impute then
# scale) and a categorical sub-pipeline (impute then one-hot encode). Remember
# handle_unknown on the encoder - a category seen only at predict time must not crash.
from pipeline import build_preprocessor, build_pipeline
numeric_cols = [c for c in X_train.select_dtypes(include=np.number).columns if c != "Pclass"]
categorical_cols = [c for c in X_train.columns if c not in numeric_cols]
print("Numeric columns:")
print(numeric_cols)

print("\nCategorical columns:")
print(categorical_cols)


Numeric columns:
['Age', 'SibSp', 'Parch', 'Fare', 'Cabin_known']

Categorical columns:
['Pclass', 'Sex', 'Embarked']


In [5]:
# TODO: build_pipeline(num_cols, cat_cols, model) - pre + model, ready to fit
baseline_model = build_pipeline(numeric_cols,categorical_cols,LogisticRegression(max_iter=1000))

### Preprocessing Design

The numerical and categorical features require different preprocessing.

For numerical columns, missing values are replaced using the median and the resulting values are standardized using `StandardScaler`.

For categorical columns, missing values are replaced using the most frequent category and the categories are converted into one-hot encoded features. `handle_unknown="ignore"` is used so that an unseen category at prediction time does not cause an error.

`Pclass` is treated as categorical even though it is stored as an integer because the values represent passenger classes rather than a continuous numerical measurement.


## Step 2: Cross-validated baseline

In [6]:
cv = StratifiedKFold(n_splits=5,shuffle=True,random_state=42)

In [7]:
# TODO: cross_val_score on the training set only, an appropriate metric for this
# target's class balance, and report BOTH the mean and the standard deviation
baseline_scores = cross_val_score(
    baseline_model,
    X_train,
    y_train,
    cv=cv,
    scoring="f1"
)
print("Baseline F1 scores:")
print(np.round(baseline_scores, 4))

print("\nMean F1:")
print(round(baseline_scores.mean(), 4))

print("\nStandard deviation:")
print(round(baseline_scores.std(), 4))

Baseline F1 scores:
[0.7317 0.6667 0.7195 0.702  0.7123]

Mean F1:
0.7064

Standard deviation:
0.0221


### Baseline Interpretation

The Logistic Regression model is used as the baseline because it provides a simple reference model for comparison with more complex models and engineered features.

The model is evaluated using F1-score through five-fold stratified cross-validation. Both the mean and standard deviation are reported because the performance can vary between folds. The mean F1 provides an estimate of overall performance, while the standard deviation indicates the amount of fold-to-fold variation.


## Step 3: Engineer features, with a stated hypothesis first

### Feature Engineering Hypothesis

I hypothesize that family structure may contain useful information about passenger survival. Passengers travelling with family members may have different survival outcomes from passengers travelling alone.

Therefore, I will test two features:

* `FamilySize`: the total number of people in the passenger's family group.
* `IsAlone`: whether the passenger was travelling alone.

I will compare their cross-validated F1 scores with the baseline to determine whether these features improve the model.


In [22]:
# TODO: engineer(df) in pipeline.py - for each feature, write the hypothesis as a
# comment before the code. Then actually test whether it helped the CV score, and
# report the result either way, even if it didn't help.
from pipeline import engineer_features
X_train_eng = engineer_features(X_train)
X_test_eng = engineer_features(X_test)
X_train_eng.head()


,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,Cabin_known,FamilySize,IsAlone
999,3,female,NaN,0,0,7.7500,Q,0,1,1
392,2,female,24.0,1,0,27.7208,C,0,2,0
628,3,female,11.0,4,2,31.2750,S,0,7,0
1165,3,male,25.0,0,0,7.2250,C,0,1,1
604,3,female,16.0,0,0,7.6500,S,0,1,1


In [23]:
X_test_eng.head()

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,Cabin_known,FamilySize,IsAlone
1028,3,female,NaN,1,0,24.1500,Q,0,2,0
1121,3,male,NaN,1,1,22.3583,C,0,3,0
1155,3,male,NaN,0,0,7.7750,S,0,1,1
1251,3,male,30.5,0,0,8.0500,S,0,1,1
721,3,male,36.0,0,0,7.4958,S,0,1,1


In [18]:
print(X_train_eng[["SibSp", "Parch", "FamilySize", "IsAlone"]].head())

      SibSp  Parch  FamilySize  IsAlone
999       0      0           1        1
392       1      0           2        0
628       4      2           7        0
1165      0      0           1        1
604       0      0           1        1


In [9]:
from pipeline import get_column_groups
X_family = X_train_eng.drop(columns=["IsAlone"])

num_family, cat_family = get_column_groups(X_family)

family_model = build_pipeline(
    num_family,
    cat_family,
    LogisticRegression(max_iter=1000)
)

family_scores = cross_val_score(
    family_model,
    X_family,
    y_train,
    cv=cv,
    scoring="f1"
)

print("FamilySize Mean F1:", family_scores.mean())
print("FamilySize Std:", family_scores.std())

FamilySize Mean F1: 0.7064403401903927
FamilySize Std: 0.02211672042055767


In [11]:
X_alone = X_train_eng.drop(columns=["FamilySize"])

num_alone, cat_alone = get_column_groups(X_alone)

alone_model = build_pipeline(
    num_alone,
    cat_alone,
    LogisticRegression(max_iter=1000)
)

alone_scores = cross_val_score(
    alone_model,
    X_alone,
    y_train,
    cv=cv,
    scoring="f1"
)

print("IsAlone Mean F1:", alone_scores.mean())
print("IsAlone Std:", alone_scores.std())

IsAlone Mean F1: 0.7026820314526206
IsAlone Std: 0.02255963810616006


In [12]:
num_eng, cat_eng = get_column_groups(X_train_eng)

both_model = build_pipeline(
    num_eng,
    cat_eng,
    LogisticRegression(max_iter=1000)
)

both_scores = cross_val_score(
    both_model,
    X_train_eng,
    y_train,
    cv=cv,
    scoring="f1"
)

print("Both features Mean F1:", both_scores.mean())
print("Both features Std:", both_scores.std())

Both features Mean F1: 0.7026820314526206
Both features Std: 0.02255963810616006


In [13]:
feature_results = pd.DataFrame({
    "Version": [
        "Baseline",
        "FamilySize",
        "IsAlone",
        "FamilySize + IsAlone"
    ],
    "Mean_F1": [
        baseline_scores.mean(),
        family_scores.mean(),
        alone_scores.mean(),
        both_scores.mean()
    ],
    "Std_F1": [
        baseline_scores.std(),
        family_scores.std(),
        alone_scores.std(),
        both_scores.std()
    ]
})

feature_results

,Version,Mean_F1,Std_F1
0,Baseline,0.706440,0.022117
1,FamilySize,0.706440,0.022117
2,IsAlone,0.702682,0.022560
3,FamilySize + IsAlone,0.702682,0.022560


### Feature Engineering Results

The `FamilySize` feature produced the same mean F1 score as the baseline, so it did not provide an improvement for Logistic Regression.

The `IsAlone` feature produced a slightly lower mean F1 score than the baseline. Using both `FamilySize` and `IsAlone` together also resulted in the same lower score.

Therefore, the hypothesis that family-based features would improve the Logistic Regression model was not supported by the cross-validation results. I have kept these results rather than reporting only features that improved the score, since the purpose of the experiment is to test the hypothesis honestly.


## Step 4: Compare at least three models

In [24]:
# TODO: cross-validate at least three different model types with the same
# preprocessing, and report mean + std for each. Are the differences bigger than the
# fold-to-fold noise?
# random forest,logistic regression, SVM

logistic_pipeline = build_pipeline(
    num_eng,
    cat_eng,
    LogisticRegression(max_iter=1000)
)

rf_pipeline = build_pipeline(
    num_eng,
    cat_eng,
    RandomForestClassifier(n_estimators=300,random_state=42,n_jobs=-1)
)

svm_pipeline = build_pipeline(
    num_eng,
    cat_eng,
    SVC()
)

In [25]:
models = {
    "Logistic Regression": logistic_pipeline,
    "Random Forest": rf_pipeline,
    "SVM": svm_pipeline
}

model_results = []

for name, model in models.items():

    scores = cross_val_score(model,X_train_eng,y_train,cv=cv,scoring="f1")

    model_results.append({"Model": name,"Mean_F1": scores.mean(),"Std_F1": scores.std()})

model_results = pd.DataFrame(model_results).sort_values(
    "Mean_F1",
    ascending=False
)

model_results

,Model,Mean_F1,Std_F1
2,SVM,0.727745,0.010555
1,Random Forest,0.714145,0.015266
0,Logistic Regression,0.702682,0.022560


### Model Comparison Results

The three models were evaluated using the same preprocessing pipeline, training data, five-fold cross-validation, and F1-score.

SVM achieved the highest mean F1 score of 0.7277, followed by Random Forest with 0.7141 and Logistic Regression with 0.7027. SVM also had the lowest standard deviation among the three models.

However, the differences between the models are relatively small compared with the fold-to-fold variation. Therefore, SVM is treated as the strongest candidate for further investigation, but the results should not be interpreted as evidence of a very large difference in model performance.

Based on these results, SVM was selected for hyperparameter tuning in the next step.



## Step 5: Tune the best model, then evaluate the test set exactly once

In [16]:
# TODO: GridSearchCV on training data only (remember the model__param prefix for
# a parameter inside a named pipeline step)
param_grid = {
    "model__C": [0.1, 1, 10, 100],
    "model__gamma": ["scale", 0.01, 0.1, 1],
    "model__kernel": ["rbf", "linear"]
}

grid = GridSearchCV(
    svm_pipeline,
    param_grid=param_grid,
    cv=cv,
    scoring="f1",
    n_jobs=-1,
    refit=True
)

grid.fit(X_train_eng, y_train)

print("Best parameters:")
print(grid.best_params_)

print("\nBest CV F1:")
print(grid.best_score_)

Best parameters:
{'model__C': 1, 'model__gamma': 0.1, 'model__kernel': 'rbf'}

Best CV F1:
0.7279971609392559


### Hyperparameter Tuning

Since SVM achieved the highest mean F1 score during the model comparison, it was selected for hyperparameter tuning.

`GridSearchCV` was applied only to the training data. The search varied `C`, `gamma`, and `kernel`. The best configuration was an RBF kernel with `C=1` and `gamma=0.1`.

The best cross-validated F1 score was approximately 0.7280. This was only a small improvement over the untuned SVM, so the tuning process did not produce a large performance gain.

The held-out test set was not used during tuning, keeping it independent for the final evaluation.


In [17]:
# TODO: the test set, touched here for the first and only time - report an
# appropriate metric. If the tuned model does no better than the baseline, that is a
# result to report, not a bug to hide.
test_predictions = grid.predict(X_test_eng)

test_f1 = f1_score( y_test, test_predictions)

print(f"Final Test F1: {test_f1:.4f}")

print(classification_report(y_test,test_predictions))

Final Test F1: 0.7812
              precision    recall  f1-score   support

           0       0.85      0.90      0.87       162
           1       0.82      0.75      0.78       100

    accuracy                           0.84       262
   macro avg       0.83      0.82      0.83       262
weighted avg       0.84      0.84      0.84       262



## Closing Analysis

Overall, the pipeline worked as expected and kept the preprocessing steps separate for numerical and categorical features. Using the pipeline also ensured that preprocessing was fitted only on the training portion during cross-validation, which helped avoid data leakage.

The Logistic Regression baseline achieved a mean F1 score of 0.7064 with a standard deviation of 0.0221. I then tested the `FamilySize` and `IsAlone` features based on the hypothesis that family structure might affect passenger survival. `FamilySize` produced the same mean F1 score as the baseline, while `IsAlone` and the combination of both features produced a slightly lower mean F1 score of 0.7027. Therefore, the feature engineering hypothesis was not supported by these results.

For model comparison, Logistic Regression achieved a mean F1 of 0.7027, Random Forest achieved 0.7141, and SVM achieved the highest mean F1 of 0.7277. SVM was therefore selected as the candidate for hyperparameter tuning. However, the differences between the models were relatively small compared with the fold-to-fold variation, so the results should not be interpreted as evidence of a very large difference in model performance.

GridSearchCV was then used on the training data to tune the SVM. The best configuration was an RBF kernel with `C=1` and `gamma=0.1`, giving a best cross-validated F1 score of approximately 0.7280.

Finally, the tuned model was evaluated on the held-out test set once. The final test F1 score was 0.7812, with an accuracy of 0.84. For the survival class, the model achieved a precision of 0.82 and recall of 0.75.

If I had more time, I would explore other feature-engineering ideas, such as extracting passenger titles from names, and investigate whether other modeling or class-weighting approaches could improve performance. I would also avoid making further tuning decisions based on the test-set result, since the test set should remain an unbiased final evaluation.
